# Motor de consultas analíticas en Colab

Entorno de GPU: compila el backend CUDA, genera TPC-H SF10, mide serial / omp / cuda y grafica.
Antes de empezar: *Entorno de ejecución → Cambiar tipo de entorno → GPU* (T4, L4 o A100).

Clonar el repo (rellenar `REPO_URL`) o, si se deja vacío, subir un `.zip` del proyecto.

In [ ]:
REPO_URL = ""  # p. ej. https://github.com/usuario/proyecto.git (un repo privado necesita un token en la URL)
import os
if REPO_URL:
    !git clone -q {REPO_URL} proyecto
else:
    from google.colab import files
    zip_subido = next(iter(files.upload()))
    !unzip -q -o "{zip_subido}" -d proyecto_zip
    # El zip puede traer la carpeta del proyecto dentro o los archivos sueltos.
    raiz = next(os.path.dirname(r) for r, _, fs in os.walk("proyecto_zip") if "Makefile" in fs and "src" in os.listdir(r))
    !rm -rf proyecto && mv "{raiz}" proyecto
%cd /content/proyecto

In [ ]:
!pip install -q duckdb numpy matplotlib
!nvidia-smi

In [ ]:
import re, subprocess
consulta = lambda campo: subprocess.check_output(["nvidia-smi", f"--query-gpu={campo}", "--format=csv,noheader"], text=True).splitlines()[0].strip()
# compute_cap da la arquitectura directamente: T4 7.5 → sm_75, A100 8.0 → sm_80, L4 8.9 → sm_89.
CUDA_ARCH = "sm_" + consulta("compute_cap").replace(".", "")
GPU = re.sub(r"^(NVIDIA|Tesla)\s+", "", consulta("name")).replace(" ", "-")
print(GPU, CUDA_ARCH)

In [ ]:
!make clean && make CUDA=1 CUDA_ARCH={CUDA_ARCH}

In [ ]:
# SF10 ocupa unos 2 GB en disco; DuckDB necesita varios GB de RAM para generarlo.
!test -d data/sf10 || python scripts/gen_datos.py 10

In [ ]:
!python scripts/bench.py --datos data/sf10 --backends serial,omp,cuda --etiqueta {GPU} --maquina {GPU}

In [ ]:
!python scripts/graficas.py resultados/*.csv
from glob import glob
from IPython.display import Image, display
for png in sorted(glob("resultados/graficas/*.png")):
    print(png)
    display(Image(png))

Descargar el CSV para versionarlo en `resultados/` del repo (junto a los de otras GPUs, `graficas.py` genera la comparación por GPU).

In [ ]:
from google.colab import files
files.download(f"resultados/{GPU}.csv")